# Hierarchical Ministry Demo

This notebook demonstrates the new lifecycle-aware agents, tool capabilities, and the
`MinistryOfExperts` coordinator. A "minister" decomposes the goal, expert agents use guarded
tools, and a cabinet chair synthesises the final recommendation.

In [ ]:
from agentic_codex import AgentBuilder, Context, FunctionAdapter, MinistryOfExperts
from agentic_codex.core.capabilities import ContextCapability, ToolCapability
from agentic_codex.core.kernel import AgentKernel
from agentic_codex.core.schemas import AgentStep, Message
from agentic_codex.core.tools import MathToolAdapter, ToolPermissions, BudgetGuard

GOAL = "Plan a weekend astronomy workshop budget with two activity tracks"

permissions = ToolPermissions({"finance": {"math"}})
budget_guard = BudgetGuard(max_calls=3)

context_capability = ContextCapability(
    data={
        "activities": [
            {"name": "Sky tour", "cost": 150},
            {"name": "Telescope lab", "cost": 220},
            {"name": "Astrophotography", "cost": 180}
        ],
        "participants": 25,
    }
)

tool_capability = ToolCapability(
    tools={"math": MathToolAdapter(name="math")}, permissions=permissions, budget=budget_guard
)

def planner_decide(ctx: Context) -> AgentStep:
    activities = ctx.scratch["context"]["activities"]
    tracks = [activity["name"] for activity in activities[:2]]
    return AgentStep(
        out_messages=[Message(role="planner", content=f"Planned tracks: {', '.join(tracks)}")],
        state_updates={"tasks": tracks},
    )

planner_kernel = AgentKernel(decide_hook=planner_decide)

planner = (
    AgentBuilder(name="minister", role="planner")
    .with_capabilities([context_capability])
    .with_step(planner_decide)
    .build(kernel=planner_kernel)
)

def expert_step(ctx: Context) -> AgentStep:
    task = ctx.scratch.get("current_task", "budget")
    activity = next(item for item in ctx.scratch["context"]["activities"] if item["name"] == task)
    tool = ctx.get_tool("math")
    per_person = tool.invoke(expression=f"{activity['cost']} / {ctx.scratch['context']['participants']}")
    message = Message(
        role="expert",
        content=f"{task}: total ${activity['cost']} | per person ${per_person['result']:.2f}",
    )
    return AgentStep(out_messages=[message])

finance_builder = (
    AgentBuilder(name="finance", role="expert")
    .with_capabilities([context_capability, tool_capability])
    .with_step(expert_step)
)

def cabinet_step(ctx: Context) -> AgentStep:
    notes = [message.content for message in ctx.inbox if message.role == "expert"]
    summary = "Cabinet summary:\n" + "\n".join(notes)
    return AgentStep(out_messages=[Message(role="cabinet", content=summary)])

cabinet = AgentBuilder(name="cabinet", role="review").with_capabilities([context_capability]).with_step(cabinet_step).build()

experts = [finance_builder.build()]
coordinator = MinistryOfExperts(planner=planner, experts=experts, cabinet=cabinet)
ctx = Context(goal=GOAL)
result = coordinator.run(goal=GOAL, inputs={})
result.messages[-1].content